In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-obp qiskit-addon-utils rustworkx
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Operator backpropagation (OBP) para sa pagtatantya ng expectation values

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Tantya sa paggamit: 4 na minuto sa isang Heron r3 processor (PAALALA: Ito ay tantya lamang. Maaaring mag-iba ang iyong runtime.)*
## Mga layunin sa pag-aaral
Pagkatapos ng tutorial na ito, dapat maunawaan ng mga gumagamit ang:
- Paano gamitin ang [`qiskit-addon-obp`](https://github.com/Qiskit/qiskit-addon-obp) upang mabawasan ang lalim ng quantum circuit sa halaga ng mas maraming circuit execution
- Paano gamitin ang [`qiskit-addon-utils`](https://github.com/Qiskit/qiskit-addon-utils) upang bumuo ng mga XYZ Hamiltonian at ang kanilang mga time-evolution circuit

## Mga kinakailangan
Iminumungkahi naming maging pamilyar ang mga gumagamit sa mga sumusunod na paksa bago gawin ang tutorial na ito:
- Paggamit ng [Estimator](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/estimator-v2) primitive upang kalkulahin ang mga expectation value ng isang observable

## Background
Ang operator backpropagation ay isang teknikang nagsasangkot ng pag-absorb ng mga operasyon mula sa dulo ng isang quantum circuit tungo sa sinusukat na observable, na karaniwang nagpapababa ng lalim ng circuit sa halaga ng karagdagang mga term sa observable. Ang layunin ay mag-backpropagate ng kasing dami ng circuit hangga't maaari nang hindi pinapayagang lumaki nang labis ang observable. Ang isang Qiskit-based na implementasyon ay available sa OBP Qiskit addon. Basahin ang kaukulang [dokumentasyon](https://qiskit.github.io/qiskit-addon-obp/) para sa karagdagang impormasyon.

Isaalang-alang ang isang halimbawang circuit kung saan ang isang observable $O = \sum_P c_P P$ ay susukatiin, kung saan ang $P$ ay mga Pauli at ang $c_P$ ay mga coefficient. Itakda natin ang circuit bilang isang unitary $U$ na maaaring lohikal na hatiin sa $U = U_C U_Q$ tulad ng ipinapakita sa figure sa ibaba.

![Circuit diagram showing Uq followed by Uc](../docs/images/tutorials/improving-estimation-of-expectation-values-with-operator-backpropagation/logical-partitioning.avif)

Ang operator backpropagation ay nag-aabsorb ng unitary $U_C$ sa observable sa pamamagitan ng pag-evolve nito bilang $O' = U_C^{\dagger}OU_C = \sum_P c_P U_C^{\dagger}PU_C$. Sa madaling salita, ang bahagi ng computation ay ginagawa nang classical sa pamamagitan ng evolusyon ng observable mula sa $O$ tungo sa $O'$. Ang orihinal na problema ay maaari ngayong muling iformula bilang pagsukat ng observable $O'$ para sa bagong mas mababang lalim na circuit na ang unitary ay $U_Q$.

Ang unitary $U_C$ ay kinakatawan bilang isang bilang ng mga slice $U_C = U_S U_{S-1}...U_2U_1$. May maraming paraan para tukuyin ang isang slice. Halimbawa, sa halimbawang circuit sa itaas, ang bawat layer ng $R_{zz}$ at bawat layer ng $R_x$ gates ay maaaring ituring bilang indibidwal na slice. Ang backpropagation ay nagsasangkot ng pagkalkula ng $O' = \Pi_{s=1}^S \sum_P c_P U_s^{\dagger} P U_s$ nang classical. Ang bawat slice $U_s$ ay maaaring katawanin bilang $U_s = exp(\frac{-i\theta_s P_s}{2})$, kung saan ang $P_s$ ay isang $n$-qubit Pauli at ang $\theta_s$ ay isang scalar. Madaling ma-verify na

$$
U_s^{\dagger} P U_s = P \qquad \text{if} ~[P,P_s] = 0,
$$

$$
U_s^{\dagger} P U_s = \qquad cos(\theta_s)P + i sin(\theta_s)P_sP \qquad \text{if} ~{P,P_s} = 0
$$

Sa halimbawa sa itaas, kung ${P,P_s} = 0$, kailangan nating magsagawa ng dalawang quantum circuit, sa halip na isa, upang kalkulahin ang expectation value. Samakatuwid, ang backpropagation ay maaaring magdagdag ng bilang ng mga term sa observable, na nagreresulta sa mas mataas na bilang ng circuit execution. Isang paraan upang payagan ang mas malalim na backpropagation sa circuit, habang pinipigilan ang operator mula sa paglaki nang labis, ay ang mag-truncate ng mga term na may maliliit na coefficient, sa halip na idagdag ang mga ito sa operator. Halimbawa, sa halimbawa sa itaas, maaari tayong pumili na mag-truncate ng term na nagsasangkot ng $P_sP$ kung sakaling ang $\theta_s$ ay sapat na maliit. Ang pag-truncate ng mga term ay maaaring magresulta sa mas kakaunting quantum circuit na isasagawa, ngunit ang paggawa nito ay nagreresulta ng ilang error sa panghuling pagkalkula ng expectation value na proporsyonal sa magnitude ng mga coefficient ng mga na-truncate na term.
## Requirements
Bago simulan ang tutorial na ito, siguraduhing mayroon ka ng mga sumusunod na naka-install:

- Qiskit SDK v2.0 o mas bago, na may suporta sa [visualization](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 o mas bago (`pip install qiskit-ibm-runtime`)
- OBP Qiskit addon 0.3 o mas bago (`pip install qiskit-addon-obp`)
- Qiskit addon utils 0.3 o mas bago (`pip install qiskit-addon-utils`)
## Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.primitives import StatevectorEstimator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import CouplingMap
from qiskit.synthesis import LieTrotter

from qiskit_addon_utils.problem_generators import generate_xyz_hamiltonian
from qiskit_addon_utils.problem_generators import (
    generate_time_evolution_circuit,
)
from qiskit_addon_utils.slicing import slice_by_depth, combine_slices
from qiskit_addon_obp.utils.simplify import OperatorBudget
from qiskit_addon_obp import backpropagate
from qiskit_addon_obp.utils.truncating import setup_budget

from rustworkx.visualization import graphviz_draw

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2, EstimatorOptions

## Small-scale simulator example
Isinasagawa ng tutorial na ito ang isang [Qiskit pattern](/guides/intro-to-patterns) para sa pagsisimula ng quantum dynamics ng isang Heisenberg spin chain gamit ang [OBP Qiskit addon](https://github.com/Qiskit/qiskit-addon-obp). Pansinin na sa isang noiseless simulator, ang expectation value na nakuha nang may at walang backpropagation ay magiging pareho.
### Step 1: Map classical inputs to a quantum problem
#### Map the time-evolution of a quantum Heisenberg model to a quantum experiment
Una, gagamitin natin ang function na [`generate_xyz_hamiltonian`](https://docs.quantum.ibm.com/api/qiskit-addon-utils/problem-generators#generate_xyz_hamiltonian) mula sa `qiskit-addon-utils` upang bumuo ng isang Heisenberg-like Hamiltonian sa isang ibinigay na connectivity graph. Ang graph na ito ay maaaring [rustworkx.PyGraph](https://www.rustworkx.org/apiref/rustworkx.PyGraph.html) o [CouplingMap](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.CouplingMap). Sa sumusunod, gagamitin natin ang isang linear chain na `CouplingMap` ng 10 qubit.

In [2]:
num_qubits = 10
layout = [(i - 1, i) for i in range(1, num_qubits)]

# Instantiate a CouplingMap object
coupling_map = CouplingMap(layout)
graphviz_draw(coupling_map.graph, method="circo")

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/operator-back-propagation/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif)

Susunod, bubuo tayo ng isang Pauli operator na nagmomodelo ng Heisenberg XYZ Hamiltonian:

$$
{\hat{\mathcal{H}}_{XYZ} = \sum_{(j,k)\in E} (J_{x} \sigma_j^{x} \sigma_{k}^{x} + J_{y} \sigma_j^{y} \sigma_{k}^{y} + J_{z} \sigma_j^{z} \sigma_{k}^{z}) + \sum_{j\in V} (h_{x} \sigma_j^{x} + h_{y} \sigma_j^{y} + h_{z} \sigma_j^{z}),}
$$

kung saan ang $G(V,E)$ ay ang graph ng coupling map. Para sa tutorial na ito, ginamit natin ang $J_x, J_y, J_z$ na $\frac{\pi}{8}, \frac{\pi}{4}, \frac{\pi}{2}$, ayon sa pagkakasunod, at $h_x, h_y, h_z$ na $\frac{\pi}{3}, \frac{\pi}{6}, \frac{\pi}{9}$, ayon sa pagkakasunod.

In [3]:
# Get a qubit operator describing the Heisenberg XYZ model
hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(np.pi / 8, np.pi / 4, np.pi / 2),
    ext_magnetic_field=(np.pi / 3, np.pi / 6, np.pi / 9),
)
print(hamiltonian)

SparsePauliOp(['IIIIIIIXXI', 'IIIIIIIYYI', 'IIIIIIIZZI', 'IIIIIXXIII', 'IIIIIYYIII', 'IIIIIZZIII', 'IIIXXIIIII', 'IIIYYIIIII', 'IIIZZIIIII', 'IXXIIIIIII', 'IYYIIIIIII', 'IZZIIIIIII', 'IIIIIIIIXX', 'IIIIIIIIYY', 'IIIIIIIIZZ', 'IIIIIIXXII', 'IIIIIIYYII', 'IIIIIIZZII', 'IIIIXXIIII', 'IIIIYYIIII', 'IIIIZZIIII', 'IIXXIIIIII', 'IIYYIIIIII', 'IIZZIIIIII', 'XXIIIIIIII', 'YYIIIIIIII', 'ZZIIIIIIII', 'IIIIIIIIIX', 'IIIIIIIIIY', 'IIIIIIIIIZ', 'IIIIIIIIXI', 'IIIIIIIIYI', 'IIIIIIIIZI', 'IIIIIIIXII', 'IIIIIIIYII', 'IIIIIIIZII', 'IIIIIIXIII', 'IIIIIIYIII', 'IIIIIIZIII', 'IIIIIXIIII', 'IIIIIYIIII', 'IIIIIZIIII', 'IIIIXIIIII', 'IIIIYIIIII', 'IIIIZIIIII', 'IIIXIIIIII', 'IIIYIIIIII', 'IIIZIIIIII', 'IIXIIIIIII', 'IIYIIIIIII', 'IIZIIIIIII', 'IXIIIIIIII', 'IYIIIIIIII', 'IZIIIIIIII', 'XIIIIIIIII', 'YIIIIIIIII', 'ZIIIIIIIII'],
              coeffs=[0.39269908+0.j, 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j,
 0.78539816+0.j, 1.57079633+0.j, 0.39269908+0.j, 0.78539816+0.j,
 1.57079633+0.j, 0.39269908+0.j, 0.

From the qubit operator, we can generate a quantum circuit which models its time evolution. We have used [`generate_time_evolution_circuit`](/docs/api/qiskit-addon-utils/problem-generators#generate_time_evolution_circuit) with Lie Trotter decomposition to construct the time evolution circuit.

In [4]:
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=0.2,
    synthesis=LieTrotter(reps=2),
)
circuit.draw("mpl", style="iqp", fold=-1)

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/5208e0a8-0.avif" alt="Output of the previous code cell" />

Mula sa qubit operator, maaari tayong lumikha ng quantum circuit na nagmomodelo ng time evolution nito. Ginamit namin ang [`generate_time_evolution_circuit`](https://docs.quantum.ibm.com/api/qiskit-addon-utils/problem-generators#generate_time_evolution_circuit) na may Lie Trotter decomposition upang buuin ang time evolution circuit.

In [5]:
slices = slice_by_depth(circuit, max_slice_depth=1)
print(f"Separated the circuit into {len(slices)} slices.")

Separated the circuit into 18 slices.


![Output of the previous code cell](../docs/images/tutorials/operator-back-propagation/extracted-outputs/5208e0a8-0.avif)

### Step 2: Optimize problem for quantum hardware execution
#### Create circuit slices to backpropagate
Ang function na `backpropagate` ay nagba-backpropagate ng buong circuit slice nang sabay-sabay. Kaya naman, ang pagpili kung paano maghiwa ay maaaring magkaroon ng epekto sa kung gaano kahusay gumagana ang backpropagation para sa isang partikular na problema. Dito, pagsasamahin natin ang mga gate ng parehong uri sa mga slice gamit ang function na [`slice_by_depth`](https://docs.quantum.ibm.com/api/qiskit-addon-utils/slicing#slice_by_depth).

Para sa mas detalyadong talakayan tungkol sa circuit slicing, tingnan ang [how-to guide](https://qiskit.github.io/qiskit-addon-utils/how_tos/create_circuit_slices.html) na ito ng package na [`qiskit-addon-utils`](https://github.com/Qiskit/qiskit-addon-utils).

In [6]:
op_budget = OperatorBudget(max_qwc_groups=8)

#### Backpropagate slices from the circuit

First we specify the observable to be $M_Z = \frac{1}{N} \sum_{i=1}^N \langle Z_i \rangle$, $N$ being the number of qubits. We will backpropagate slices from the time-evolution circuit until the terms in the observable can no longer be combined into eight or fewer qubit-wise commuting Pauli groups.

In [7]:
observable = SparsePauliOp.from_sparse_list(
    [("Z", [i], 1 / num_qubits) for i in range(num_qubits)],
    num_qubits=num_qubits,
)
observable

SparsePauliOp(['IIIIIIIIIZ', 'IIIIIIIIZI', 'IIIIIIIZII', 'IIIIIIZIII', 'IIIIIZIIII', 'IIIIZIIIII', 'IIIZIIIIII', 'IIZIIIIIII', 'IZIIIIIIII', 'ZIIIIIIIII'],
              coeffs=[0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j, 0.1+0.j,
 0.1+0.j, 0.1+0.j])

#### Constrain how large the operator can grow during backpropagation
Sa panahon ng backpropagation, ang bilang ng mga term sa operator ay karaniwang lalapit sa $2^L$ nang mabilis, kung saan ang $L$ ay ang bilang ng mga slice. Kapag ang dalawang term sa operator ay hindi nag-commute nang qubit-wise, kailangan natin ng hiwalay na mga circuit upang makuha ang mga expectation value na tumutugma sa kanila. Halimbawa, kung mayroon tayong dalawang-qubit observable na $O = 0.1 XX + 0.3 IZ - 0.5 IX$, dahil ang $[XX,IX] = 0$, ang pagsukat sa isang basis lamang ay sapat upang kalkulahin ang mga expectation value para sa dalawang term na ito. Gayunpaman, ang $IZ$ ay anti-commute sa dalawang term, kaya kailangan natin ng hiwalay na basis measurement upang kalkulahin ang expectation value ng $IZ$. Sa madaling salita, kailangan natin ng dalawang circuit sa halip na isa upang kalkulahin ang $\langle O \rangle$. Habang dumarami ang bilang ng mga term sa operator, may posibilidad na dumarami rin ang kinakailangang bilang ng circuit execution.

Ang laki ng operator ay maaaring limitahan sa pamamagitan ng pagtukoy ng `operator_budget` kwarg ng function na `backpropagate`, na tumatanggap ng isang instance ng [OperatorBudget](https://docs.quantum.ibm.com/api/qiskit-addon-obp/utils-simplify#operatorbudget).

Upang kontrolin ang dami ng karagdagang resources (bilang ng circuit execution, at samakatuwid ang kinakailangang QPU time) na inilalaan, pinapaghihigpitan natin ang maximum na bilang ng qubit-wise commuting Pauli groups na pinapayagang magkaroon ang backpropagated observable. Dito, tinukoy natin na ang backpropagation ay dapat tumigil kapag ang bilang ng qubit-wise commuting Pauli groups sa operator ay lumampas sa walo.

In [ ]:
# Backpropagate slices onto the observable
bp_obs, remaining_slices, metadata = backpropagate(
    observable, slices, operator_budget=op_budget
)
# Recombine the slices remaining after backpropagation
bp_circuit = combine_slices(remaining_slices)

print(f"Backpropagated {metadata.num_backpropagated_slices} slices.")
print(
    f"New observable has {len(bp_obs.paulis)} terms, which can be combined into "
    f"{len(bp_obs.group_commuting(qubit_wise=True))} groups."
)
print(
    f"Note that backpropagating one more slice would result in "
    f"{metadata.backpropagation_history[-1].num_paulis[0]} terms "
    f"across {metadata.backpropagation_history[-1].num_qwc_groups} groups."
)
print("The remaining circuit after backpropagation looks as follows:")
bp_circuit.draw("mpl", fold=-1, scale=0.6)

Backpropagated 6 slices.
New observable has 60 terms, which can be combined into 6 groups.
Note that backpropagating one more slice would result in 114 terms across 12 groups.
The remaining circuit after backpropagation looks as follows:


<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/ee8fd385-1.avif" alt="Output of the previous code cell" />

#### Backpropagate slices from the circuit
Una, tinukoy natin ang observable bilang $M_Z = \frac{1}{N} \sum_{i=1}^N \langle Z_i \rangle$, kung saan ang $N$ ay ang bilang ng mga qubit. Mag-backpropagate tayo ng mga slice mula sa time-evolution circuit hanggang sa ang mga term sa observable ay hindi na maaaring pagsamahin sa walong o mas kaunting qubit-wise commuting Pauli groups.

In [ ]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)
print(backend)

<IBMBackend('ibm_kingston')>


In [10]:
pm_basis = generate_preset_pass_manager(
    optimization_level=3, basis_gates=backend.configuration().basis_gates
)
isa_circuit = pm_basis.run(circuit)
isa_bp_circuit = pm_basis.run(bp_circuit)

Sa ibaba, makikita mo na nag-backpropagate tayo ng anim na slice, at ang mga term ay pinagsama sa anim at hindi walong grupo. Ito ay nagpapahiwatig na ang pag-backpropagate pa ng isang slice ay magiging sanhi ng paglampas ng bilang ng mga Pauli group sa walong. Maaari nating i-verify na ito ang kaso sa pamamagitan ng pagsusuri sa ibinalik na metadata. Pansinin din na sa bahaging ito, ang circuit transformation ay eksakto. Ibig sabihin, walang mga term ng bagong observable na $O'$ ang na-truncate. Ang backpropagated circuit at ang backpropagated operator ay nagbibigay ng eksaktong kinalabasan tulad ng orihinal na circuit at operator.

In [11]:
pubs = [(isa_circuit, observable), (isa_bp_circuit, bp_obs)]

In [12]:
rng = np.random.default_rng()
estimator = StatevectorEstimator(seed=rng)
job = estimator.run(pubs)

![Output of the previous code cell](../docs/images/tutorials/operator-back-propagation/extracted-outputs/ee8fd385-1.avif)

Para sa small-scale na halimbawa sa isang simulator, hindi tayo gagamit ng truncation. Ito ay dahil sa kawalan ng ingay, ang circuit na may at walang backpropagation ay nagdudulot ng parehong resulta, at ang truncation ay nagpapalala ng resulta dahil sa dagdag na approximation.
#### Transpile the circuits into the basis gate set
Ngayon, ini-transpile natin ang parehong orihinal at backpropagated na mga circuit sa basis gate ng backend. Hindi natin kailangang mag-transpile sa aktwal na backend dahil magpapatakbo tayo sa isang simulator para sa maliit na instance.

In [13]:
primitive_result = job.result()
circuit_expval = primitive_result[0].data.evs.item()
bp_circuit_expval = primitive_result[1].data.evs.item()

In [14]:
methods = [
    "No backpropagation",
    "Backpropagation",
]
values = [circuit_expval, bp_circuit_expval]

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
ax.set_ylim([0.6, 0.92])
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/fb5f955a-1.avif" alt="Output of the previous code cell" />

As expected, the two expectation values agree. Because we are running on a noiseless statevector simulator, backpropagation is an exact transformation of the circuit-observable pair, so the original and backpropagated workflows must produce the same value of $M_Z$. The benefit of backpropagation only becomes apparent on noisy hardware, where the shorter backpropagated circuit accumulates less error, as illustrated in the large-scale hardware example below.

## Large-scale hardware example

When developing an experiment, it's useful to start with a small circuit to make visualizations and simulations easier. Now we look into operator backpropagation for a 50-qubit Heisenberg Hamiltonian with the same set of values for the $J$ and $h$ parameters and the same observable $M_Z$, but for four Trotter steps. The ideal expectation value at this scale cannot be calculated in a brute force method, so we use a tensor network and obtain the ideal expectation value to be $\simeq 0.89$.

Along with backpropagation, in this large-scale example, we also introduce backpropagation with truncation. Ideally we want to backpropagate as much as possible to reduce the depth of the effective circuit. However, it often leads to a large number of non-commuting terms in the updated observable, increasing the quantum overhead. Therefore, we can eliminate observable terms with small coefficients using a practice called truncation. While truncation allows more propagation by reducing the number of terms in the updated observable, it also introduces some approximation. Therefore, it is necessary to restrict the truncation within some limits so that the approximation error does not overwhelm the reduction in noise obtained from deeper backpropagation.

To restrict the amount of truncation, we allot an error budget for each slice as well as the total error budget over the entire backpropagated circuit using the [`setup_budget`](/docs/api/qiskit-addon-obp/utils-truncating#setup_budget) function. This ensures that the truncation is controlled for each slice as well as for the entire circuit. See also this [guide](https://qiskit.github.io/qiskit-addon-obp/how_tos/truncate_operator_terms.html) for other ways of allocating the budget.

In [ ]:
num_qubits = 50
layout = [(i - 1, i) for i in range(1, num_qubits)]

# Instantiate a CouplingMap object
coupling_map = CouplingMap(layout)

hamiltonian = generate_xyz_hamiltonian(
    coupling_map,
    coupling_constants=(np.pi / 8, np.pi / 4, np.pi / 2),
    ext_magnetic_field=(np.pi / 3, np.pi / 6, np.pi / 9),
)

# Generate a time evolution circuit for the Hamiltonian
circuit = generate_time_evolution_circuit(
    hamiltonian,
    time=0.2,
    synthesis=LieTrotter(reps=4),
)

# Define the observable to measure
observable = SparsePauliOp.from_sparse_list(
    [("Z", [i], 1 / num_qubits) for i in range(num_qubits)],
    num_qubits,
)

slices = slice_by_depth(circuit, max_slice_depth=1)

# Define the maximum number of qwc groups allowed in the
# backpropagated observable,
# and the truncation error budget
op_budget = OperatorBudget(max_qwc_groups=15)
truncation_error_budget = setup_budget(
    max_error_total=0.03, max_error_per_slice=0.005
)

# First backpropagation without truncation
bp_obs, remaining_slices, metadata = backpropagate(
    observable, slices, operator_budget=op_budget
)
bp_circuit = combine_slices(remaining_slices)

# Now backpropagate with truncation, using the same operator budget and
# the defined truncation error budget
bp_obs_trunc, remaining_slices_trunc, metadata = backpropagate(
    observable,
    slices,
    operator_budget=op_budget,
    truncation_error_budget=truncation_error_budget,
)
bp_circuit_trunc = combine_slices(
    remaining_slices_trunc, include_barriers=False
)

# Now we transpile the original circuit and the two backpropagated circuits,
# and apply the layout to the corresponding observables
pm = generate_preset_pass_manager(optimization_level=3, backend=backend)

isa_circuit = pm.run(circuit)
isa_bp_circuit = pm.run(bp_circuit)
isa_bp_circuit_trunc = pm.run(bp_circuit_trunc)

isa_observable = observable.apply_layout(isa_circuit.layout)
isa_bp_observable = bp_obs.apply_layout(isa_bp_circuit.layout)
isa_bp_observable_trunc = bp_obs_trunc.apply_layout(
    isa_bp_circuit_trunc.layout
)

# Compare the 2-qubit depth of each transpiled circuit to see how much
# depth backpropagation saved
print(
    f"2-qubit depth without backpropagation: "
    f"{isa_circuit.depth(lambda x: x.operation.num_qubits == 2)}"
)
print(
    f"2-qubit depth with backpropagation: "
    f"{isa_bp_circuit.depth(lambda x: x.operation.num_qubits == 2)}"
)
print(
    f"2-qubit depth with backpropagation and truncation: "
    f"{isa_bp_circuit_trunc.depth(lambda x: x.operation.num_qubits == 2)}"
)

pubs = [
    (isa_circuit, isa_observable),
    (isa_bp_circuit, isa_bp_observable),
    (isa_bp_circuit_trunc, isa_bp_observable_trunc),
]

# Now we instantiate the Estimator primitive for the hardware with
# ZNE and measurement error
# mitigation and compute the three circuits and observables
options = EstimatorOptions()
options.default_precision = 0.01
options.resilience_level = 2
options.resilience.zne.noise_factors = [1, 1.2, 1.4]
options.resilience.zne.extrapolator = ["linear"]
estimator = EstimatorV2(mode=backend, options=options)

estimator.options.environment.job_tags = ["TUT_OBP"]
job = estimator.run(pubs)

# Retrieve the results and the standard deviations
result_no_bp = job.result()[0].data.evs.item()
result_bp = job.result()[1].data.evs.item()
result_bp_trunc = job.result()[2].data.evs.item()

std_no_bp = job.result()[0].data.stds.item()
std_bp = job.result()[1].data.stds.item()
std_bp_trunc = job.result()[2].data.stds.item()

2-qubit depth without backpropagation: 24
2-qubit depth with backpropagation: 20
2-qubit depth with backpropagation and truncation: 18


In [16]:
print(f"Expectation value without backpropagation: {result_no_bp}")
print(f"Backpropagated expectation value: {result_bp}")
print(f"Backpropagated expectation value with truncation: {result_bp_trunc}")

Expectation value without backpropagation: 0.9543907942381811
Backpropagated expectation value: 0.9445337385406468
Backpropagated expectation value with truncation: 0.934050286970965


In [17]:
# Plot the results
methods = [
    "No backpropagation",
    "Backpropagation",
    "Backpropagation w/ truncation",
]
values = [result_no_bp, result_bp, result_bp_trunc]
error_bars = [std_no_bp, std_bp, std_bp_trunc]

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.errorbar(methods, values, yerr=error_bars, fmt="o", color="r", capsize=5)
plt.axhline(0.89)
ax.set_ylim([0.8, 0.98])
plt.text(0.25, 0.895, "Exact result")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/operator-back-propagation/extracted-outputs/37834c72-1.avif" alt="Output of the previous code cell" />

### Step 4: Post-process and return result in desired classical format
Ngayon nakukuha natin ang mga expectation value ng orihinal at backpropagated na mga circuit.